# RDKit 2D Descriptor Calculation (LigF)

## Purpose
Compute RDKit 2D molecular descriptors (ligand features, LigF) from SMILES strings.
These descriptors constitute the LigF feature set used in `DockF+LigF` and
`DockF+LigF+MDF` model configurations.

## Input
- CSV table with a `smiles` column (one SMILES string per docking pose row)

## Output
- Input table with RDKit 2D descriptor columns appended

## Notes
- Descriptors are computed once per unique SMILES string and broadcast to all rows
  sharing the same SMILES (multiple poses of the same ligand).
- Molecules that cannot be parsed are assigned NaN for all descriptors.
- The full descriptor list (`Descriptors._descList`) is used, plus a set of
  explicitly requested physicochemical properties (MolWt, TPSA, HBD/HBA counts, etc.).
- Duplicate descriptor names are removed; all values are cast to float.

In [ ]:
# ================================================================
# Cell 1: Libraries
# ================================================================

import os
import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors
from rdkit.Chem import rdMolDescriptors as rdmd


In [ ]:
# ================================================================
# Cell 2: Configuration
# ================================================================

# ── Configuration ──────────────────────────────────────────────────────────
# Update paths to match your local directory structure before running.
IN_CSV  = "/path/to/input.csv"
OUT_CSV = "/path/to/output.csv"

SILENCE_RDKIT_WARNINGS = True  # suppress RDKit parsing warnings
# ── End Configuration ──────────────────────────────────────────────────────


In [ ]:
# ================================================================
# Cell 3: Utilities and descriptor definitions
# ================================================================

if SILENCE_RDKIT_WARNINGS:
    RDLogger.DisableLog("rdApp.warning")

# ── Descriptor list ─────────────────────────────────────────────────────────
# Start from the full RDKit descriptor list and add explicit physicochemical
# properties to ensure key drug-likeness metrics are always present.
base_desc = list(Descriptors._descList)  # (name, func) tuples
extra_desc = [
    ("MolWt",             Descriptors.MolWt),
    ("TPSA",              rdmd.CalcTPSA),
    ("NumHBD",            rdmd.CalcNumHBD),
    ("NumHBA",            rdmd.CalcNumHBA),
    ("NumRotatableBonds", rdmd.CalcNumRotatableBonds),
    ("NumAromaticRings",  rdmd.CalcNumAromaticRings),
    ("NumAliphaticRings", rdmd.CalcNumAliphaticRings),
    ("FractionCSP3",      rdmd.CalcFractionCSP3),
]
desc_funcs = []
seen = set()
for name, func in base_desc + extra_desc:
    if name not in seen:
        seen.add(name)
        desc_funcs.append((name, func))

print(f"Total descriptors to compute: {len(desc_funcs)}")

# ── SMILES → Mol conversion ─────────────────────────────────────────────────
def smiles_to_mol(smi: str):
    """Parse SMILES to RDKit Mol; returns None on failure."""
    if not isinstance(smi, str) or not smi.strip():
        return None
    mol = Chem.MolFromSmiles(smi, sanitize=True)
    if mol is None:
        # Retry with explicit sanitization step
        mol2 = Chem.MolFromSmiles(smi, sanitize=False)
        if mol2 is not None:
            try:
                Chem.SanitizeMol(mol2)
                mol = mol2
            except Exception:
                mol = mol2
    return mol

# ── Descriptor computation ──────────────────────────────────────────────────
def compute_descriptors(mol) -> dict:
    """Compute all descriptors for a valid Mol object."""
    row = {}
    for name, func in desc_funcs:
        try:
            row[name] = func(mol)
        except Exception:
            row[name] = np.nan
    return row


In [ ]:
# ================================================================
# Cell 4: Compute descriptors
# ================================================================

# Load input
df = pd.read_csv(IN_CSV)
if "smiles" not in df.columns:
    raise ValueError("Input CSV must have a 'smiles' column.")
print(f"Input rows: {len(df)}")

# Compute per unique SMILES (avoid recomputing for identical ligands)
smiles_list = df["smiles"].astype(str).fillna("").tolist()
unique_smiles = sorted(set(smiles_list))
print(f"Unique SMILES: {len(unique_smiles)}")

nan_row = {name: np.nan for name, _ in desc_funcs}
desc_cache = {}
failed = 0
for smi in unique_smiles:
    mol = smiles_to_mol(smi)
    if mol is None:
        desc_cache[smi] = nan_row.copy()
        failed += 1
    else:
        desc_cache[smi] = compute_descriptors(mol)

print(f"Molecules parsed successfully: {len(unique_smiles) - failed}/{len(unique_smiles)}")

# Build descriptor DataFrame in original row order
desc_df = pd.DataFrame([desc_cache[s] for s in smiles_list])
desc_df = desc_df.loc[:, ~desc_df.columns.duplicated()]  # drop duplicate columns

# Cast all descriptor columns to float
for col in desc_df.columns:
    if desc_df[col].dtype == "object":
        desc_df[col] = pd.to_numeric(desc_df[col], errors="coerce")

print(f"Descriptor columns: {len(desc_df.columns)}")

# Insert descriptor columns after 'pocket_score_max'
if "pocket_score_max" in df.columns:
    insert_at = list(df.columns).index("pocket_score_max") + 1
else:
    insert_at = len(df.columns)
    print("[WARN] 'pocket_score_max' column not found; appending descriptors at the end.")

left  = df.iloc[:, :insert_at]
right = df.iloc[:, insert_at:]
out_df = pd.concat([left, desc_df, right], axis=1)


In [ ]:
# ================================================================
# Cell 5: Save output
# ================================================================

# Save
os.makedirs(os.path.dirname(OUT_CSV) or ".", exist_ok=True)
out_df.to_csv(OUT_CSV, index=False)
print(f"[OK] Saved: {OUT_CSV}  shape={out_df.shape}")
